# 04. Các Kỹ Thuật Cải Thiện
**Nội dung:**
- Chống overfitting cho SimpleRNN dùng Bidirectional + Dropout + EarlyStopping + Class Weights.
- Tăng cường dữ liệu (SMOTE) cho Logistic Regression vàNaive Bayes.
- Hyperparameter tuning cho Decision Tree và SVM.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, f1_score)

df = pd.read_csv('../final_dataset_with_all_features_v3.1.csv')

# ML pipeline
df_tab = df.drop(columns=['url', 'type', 'domain', 'scan_date'])
df_tab = df_tab.fillna(df_tab.median(numeric_only=True))
X = df_tab.drop(columns=['label'])
y = df_tab['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

# DL pipeline
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
MAX_VOCAB, MAX_LEN = 10000, 200
df_seq = df[['url', 'label']].copy()
tokenizer = Tokenizer(num_words=MAX_VOCAB, char_level=True, oov_token='<OOV>')
tokenizer.fit_on_texts(df_seq['url'])
X_seq = pad_sequences(tokenizer.texts_to_sequences(df_seq['url']),
                      maxlen=MAX_LEN, padding='post', truncating='post')
y_seq = df_seq['label'].values
X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=42, stratify=y_seq)

print(f"ML-Train: {X_train.shape} | Test: {X_test.shape}")
print(f"DL-Train: {X_train_seq.shape} | Test: {X_test_seq.shape}")

## 2. Hàm Tiện Ích

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report, confusion_matrix)

def quick_report(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    print(f"\n--- {name} ---")
    print(f"Accuracy: {acc*100:.2f}% | F1-weighted: {f1*100:.2f}% | F1-macro: {f1_macro*100:.2f}%")
    return {'Mô hình': name, 'Accuracy': acc, 'F1-weighted': f1, 'F1-macro': f1_macro}

def compare_before_after(before_dict, after_dict):
    df = pd.DataFrame([before_dict, after_dict])
    for col in ['Accuracy', 'F1-weighted', 'F1-macro']:
        df[col] = df[col].apply(lambda x: f"{x*100:.2f}%")
    print(df.to_string(index=False))
    return df

# Lưu kết quả tất cả kỹ thuật cải thiện
improvement_results = []

## 3. Chống Overfitting dùng cho SimpleRNN

Ở lần đánh giá mô hình **02_ML_models** cho biết RNN baseline đạt F1-macro 21.10%, gần như bỏ qua hoàn toàn Phishing và Malware. Có 2 nguyên nhân chính dẫn đến vấn đề trên như sau:
- Vanishing gradient khi xử lý các chuỗi khoảng 200 ký tự.
- Tình trạng mất cân bằng lớp khiến model thiên về lớp Benign.

Để cải thiện vấn đề trên ta áp dụng 4 kỹ thuật, mỗi kỹ thuật sẽ giải quyết một vấn đề cụ thể:
- **Bidirectional** xử lý chuỗi 2 chiều.
- **Dropout** chống overfitting.
- **Early Stopping** dừng sớm tại trọng số tốt nhất.
- **Class Weights** buộc model chú ý đến lớp thiểu số.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Bidirectional, Embedding, Dropout, Dense 
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

class_weights = compute_class_weight('balanced', classes=np.unique(y_train_seq), y=y_train_seq)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

before = quick_report('RNN (gốc)', y_test_seq, y_pred_rnn)

print("\nĐang huấn luyện RNN (cải thiện)...")
rnn_improved = Sequential([
    Embedding(input_dim=MAX_VOCAB, output_dim=64, input_length=MAX_LEN),
    Bidirectional(SimpleRNN(64, return_sequences=False)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(4, activation='softmax')
])

rnn_improved.compile(optimizer='adam',
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])

rnn_improved.fit(X_train_seq, y_train_seq,
                 epochs=20, batch_size=256,
                 validation_split=0.2,
                 class_weight=class_weight_dict,
                 callbacks=[early_stopping],
                 verbose=1)

y_pred_rnn_improved = rnn_improved.predict(X_test_seq).argmax(axis=1)

after = quick_report('RNN (cải thiện)', y_test_seq, y_pred_rnn_improved)
improvement_results.extend([before, after])

print("\n=== SO SÁNH TRƯỚC/SAU ===")
compare_before_after(before, after)

print("\n=== Classification Report ===")
print(classification_report(y_test_seq, y_pred_rnn_improved,
      target_names=['Benign','Defacement','Phishing','Malware']))

## 4. Tăng Cường Dữ Liệu (SMOTE)

### 4.1 Logistic Regression

Logistic Regression đạt 84.37% với recall Phishing chỉ 13%. Thử SMOTE để cân bằng phân phối lớp xem có cải thiện được không.

### 4.2 Naive Bayes

Naive Bayes có recall Phishing thấp nhất trong tất cả model (9%). Áp dụng SMOTE tương tự để kiểm tra.